# Turn the Loop Dial: SenseNova-Loop-9B-Preview (1–4 Passes) vs Plain Qwen3.5-9B

**One plain-English question:** `sensenova/SenseNova-Loop-9B-Preview` takes the 32-layer `Qwen3.5-9B` architecture and runs its **middle 24 layers (`4..27`) twice per token** by default (`4 + 24×2 + 4 = 56` layer executions), stitched together with two-stream Manifold-Constrained Hyper-Connections (`mHC`).

What happens at inference time if you turn that **loop dial** to **1, 2, 3, or 4 passes** (`32, 56, 80, or 104` layer steps per token) without retraining—and is it actually more accurate (or faster per GPU-hour) than just running plain **`Qwen/Qwen3.5-9B`** (`32` layers once)?

### Reproducibility Contract
- **Benchmarks**:
  1. **`MATH-500`** (`HuggingFaceH4/MATH-500`, `test` split), fixed `seed = 0`, nested subset `S_5 ⊂ S_50 ⊂ S_200` (`data/subset_200.json`), `16,384` max new tokens.
  2. **`AIME 2024 + 2025`** (`HuggingFaceH4/aime_2024` + `MathArena/aime_2025`, `N = 60` olympiad items in `data/aime_2024_2025.json`), `32,768` max new tokens.
- **Decoding**: `enable_thinking=True`, `temperature=1.0`, `top_p=0.95`, `top_k=20`, `min_p=0.0`, `presence_penalty=1.5`, `seed=0`, `1` sample/problem, plus 512-token Pass-2 forced answer when hitting the cap.
- **Serving Backend**: Unified `vllm==0.33.0` + upstream PR `vllm-project/vllm#64832` (`SenseNovaLoopForConditionalGeneration` + `hf_overrides["loop_settings"]`) in an isolated `/content/vllm-venv` on a Colab `>= 80 GB` GPU (`G4` RTX PRO 6000 96 GB or `A100` 80 GB) in `bfloat16` (preserving FP32 `mhc_attn` / `mhc_ffn` routers).

## 1. Clone Repository & Install Pinned Dependencies
We pin `transformers==5.18.0` (matching `SenseNova-Loop-9B-Preview`'s `config.json` for Phase A PyTorch hook checks) alongside `fla-core` for Triton GatedDeltaNet kernels, without overwriting Colab's CUDA-matched `torch`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if not Path("/content/loop-dial").exists() and Path("/content").exists():
    subprocess.run(["git", "clone", "https://github.com/DavidFlorez/loop-dial.git", "/content/loop-dial"], check=True)

REPO_ROOT = Path("/content/loop-dial") if Path("/content/loop-dial").exists() else Path.cwd().resolve().parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
constraint = Path("/tmp/torch_pin.txt")
constraint.write_text(f"torch=={torch.__version__}\n")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-c", str(constraint),
    "transformers==5.18.0", "accelerate", "safetensors",
    "datasets", "math-verify[antlr4_13_2]", "fla-core", "einops",
    "scipy", "pandas", "matplotlib", "gradio", "-e", "."
], check=True)

subprocess.run(["nvidia-smi"], check=False)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

## 2. Verify Fixed `MATH-500` (`N=200`) & `AIME 2024+2025` (`N=60`) Subsets
`data/subset_200.json` and `data/aime_2024_2025.json` freeze the exact problem lists and `unique_id` SHA-256 digests so every condition is evaluated on the exact same items.

In [ ]:
import json
from loop_dial.common import DATA_DIR, load_subset_from_disk
from loop_dial.metrics import DEFAULT_SAMPLING

subset_200 = load_subset_from_disk(n=200)
aime_payload = json.loads((DATA_DIR / "aime_2024_2025.json").read_text())
aime_60 = aime_payload["problems"]

print(f"Loaded {len(subset_200)} deterministic MATH-500 problems.")
print("First 5 MATH-500 sanity IDs:", [p["unique_id"] for p in subset_200[:5]])
print(f"Loaded {len(aime_60)} AIME 2024+2025 olympiad problems (SHA256={aime_payload['sorted_unique_ids_sha256'][:16]}...).")
print("Shared sampling config:", DEFAULT_SAMPLING)

## 3. Phase A — Verify the Loop Dial with PyTorch Forward Hooks (`32 / 56 / 80 / 104` Layer Calls)
`SenseNova-Loop-9B-Preview`'s `execution_plan(config)` requires **both** `cfg.loop_settings["eval_count"] = n` and `cfg.loop_settings["counts"] = [n]`, while keeping `mhc_attn` and `mhc_ffn` in `float32`. Below we run `scripts/smoke_test.py` (or load saved `results/smoke.json`) to verify physical decoder-layer calls per generated token over 64 tokens and inspect the 5-problem sanity check.

In [1]:
import json
from loop_dial.common import RESULTS_DIR

RUN_PHASE_A_LIVE = False  # Set True on Colab GPU to re-run Phase A from scratch
if RUN_PHASE_A_LIVE:
    subprocess.run([sys.executable, "scripts/smoke_test.py"], check=True)

smoke_data = json.loads((RESULTS_DIR / "smoke.json").read_text())
print(f"GPU: {smoke_data['gpu_name']} ({smoke_data['gpu_total_gb']} GB)")
print(f"Versions: {smoke_data['versions']}\n")
print("64-token forward-hook decoder-layer verification:")
for row in smoke_data["scaling_64_tokens"]:
    print(
        f"  {row['condition']:10s} | plan_len={row['plan_len']:3d} | "
        f"calls/tok={row['decoder_layer_calls_per_generated_token']:5.1f} | "
        f"{row['ms_per_token']:7.2f} ms/tok ({row['tokens_per_s']:6.2f} tok/s) | "
        f"peak_mem={row['peak_gpu_mem_gb']:.2f} GB"
    )

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB)
Versions: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'transformers': '5.18.0', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'datasets': '4.8.5', 'math_verify': 'installed', 'fla': '0.6.0', 'causal_conv1d': 'not_installed', 'vllm': 'not_installed'}

64-token forward-hook decoder-layer verification:
  loop_n1    | plan_len= 32 | calls/tok= 32.0 |   95.14 ms/tok ( 10.51 tok/s) | peak_mem=16.76 GB
  loop_n2    | plan_len= 56 | calls/tok= 56.0 |  184.36 ms/tok (  5.42 tok/s) | peak_mem=16.80 GB
  loop_n3    | plan_len= 80 | calls/tok= 80.0 |  270.59 ms/tok (  3.70 tok/s) | peak_mem=16.84 GB
  loop_n4    | plan_len=104 | calls/tok=104.0 |  359.29 ms/tok (  2.78 tok/s) | peak_mem=16.87 GB
  qwen35_9b  | plan_len= 32 | calls/tok= 32.0 |   18.40 ms/tok ( 54.35 tok/s) | peak_mem=33.44 GB


## 4. Install `vllm==0.33.0` + SenseNova-Loop PR `#64832` (Isolated Venv)
Stock `vllm==0.33.0` does not yet ship `SenseNovaLoopForConditionalGeneration`. `scripts/probe_vllm.py --setup-only` creates an isolated `/content/vllm-venv` (leaving Colab's base `torch` and `transformers==5.18.0` untouched), installs `vllm==0.33.0`, applies upstream [`vllm-project/vllm#64832`](https://github.com/vllm-project/vllm/pull/64832), and verifies `hf_overrides["loop_settings"]` across `n = 1, 2, 3, 4` (`32 / 56 / 80 / 104` layer steps).

`ensure_vllm_python()` is also called automatically whenever you flip `RUN_PHASE_B_LIVE = True` or `RUN_AIME_LIVE = True` below.

In [ ]:
RUN_VLLM_SETUP = False  # Set True to install/verify /content/vllm-venv now (also auto-runs if any RUN_*_LIVE=True)
VLLM_SETUP_VERIFIED = False

def ensure_vllm_python() -> str:
    global VLLM_SETUP_VERIFIED
    if not VLLM_SETUP_VERIFIED:
        subprocess.run([sys.executable, "scripts/probe_vllm.py", "--setup-only"], check=True)
        VLLM_SETUP_VERIFIED = True
    venv_py = Path("/content/vllm-venv/bin/python")
    return str(venv_py) if venv_py.exists() else sys.executable

if RUN_VLLM_SETUP:
    active_vllm_py = ensure_vllm_python()
    print("Ready vLLM Python:", active_vllm_py)
else:
    print("Skipping live vLLM install (offline replay mode). Flip RUN_VLLM_SETUP=True or any RUN_*_LIVE=True on Colab GPU.")

## 5. Phase B — Main `MATH-500` Sweep (`N = 200`, `16,384` Max New Tokens)
`scripts/run_sweep.py` is fully resumable: it skips any `unique_id` already saved in `results/<condition>.jsonl`, evaluates nested prefix budgets (`2,048 / 4,096 / 8,192 / 16,384` tokens), and runs a 512-token Pass-2 forced answer (`</think>\nFinal Answer: \boxed{`) whenever a trace hits the cap without emitting a final boxed answer.

In [ ]:
RUN_PHASE_B_LIVE = False  # Set True on Colab GPU to run or resume the MATH-500 sweep
if RUN_PHASE_B_LIVE:
    vllm_py = ensure_vllm_python()
    subprocess.run([
        vllm_py, "scripts/run_sweep.py",
        "--backend", "vllm",
        "--n-problems", "200",
        "--max-new-tokens", "16384",
        "--conditions", "loop_n2", "loop_n1", "loop_n3", "loop_n4", "qwen35_9b",
    ], check=True)

## 6. Phase C — `MATH-500` Paired McNemar Tests, Wilson 95% CIs, & Charts
Because every model condition is evaluated on the **exact same problems**, we use two-sided **exact McNemar tests** on discordant pairs (`gained` vs `lost` problems) alongside 95% Wilson confidence intervals.

In [ ]:
import pandas as pd

subprocess.run([sys.executable, "scripts/stats.py"], check=True)
summary_df = pd.read_csv("results/summary.csv")
paired_df = pd.read_csv("results/paired_stats.csv")

display_cols = [
    "label", "layer_executions_per_token", "n_problems",
    "accuracy_strict_pct", "accuracy_forced_pct",
    "ci95_strict_low_pct", "ci95_strict_high_pct",
    "hit_cap_pct", "loop_pct", "median_tokens", "correct_strict_per_gpu_hour"
]
print(summary_df[display_cols].to_string(index=False))
print("\nHeadline Paired Exact McNemar Comparisons (strict metric):")
headline = paired_df[paired_df["metric"] == "strict"]
print(headline[["comparison", "diff_cmp_minus_ref_pp", "gained_by_cmp", "lost_by_cmp", "mcnemar_p_value"]].to_string(index=False))

### Logged `MATH-500` `N = 200` Results Snapshot (from `results/summary.csv`)

| Dial setting | Layer steps / token | Solved cleanly (`≤16k` tokens) | With cap forced answer | Hit 16k cap | Median tokens | Strict answers / GPU-hr |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Loop 1 pass (32 layers) | 32 | **0.5%** (1/200, CI [0.09%, 2.78%]) | **3.5%** (7/200) | 72.5% | 16,384 | 2.1 |
| Loop 2 passes (56 layers, default) | 56 | **99.5%** (199/200, CI [97.22%, 99.91%]) | **99.5%** (199/200) | 0.0% | 572 | 665.8 |
| Loop 3 passes (80 layers) | 80 | **93.5%** (187/200, CI [89.2%, 96.16%]) | **95.5%** (191/200) | 4.5% | 685 | 370.6 |
| Loop 4 passes (104 layers) | 104 | **88.5%** (177/200, CI [83.34%, 92.21%]) | **92.5%** (185/200) | 9.0% | 856 | 228.5 |
| Qwen3.5-9B (32 layers, no loop) | 32 | **85.5%** (171/200, CI [79.95%, 89.71%]) | **96.0%** (192/200) | 14.5% | 5,230 | 581.3 |

#### Paired Exact McNemar Tests (`MATH-500`, `N = 200` Identical Problems, Two-Sided)

| Comparison | Metric | Accuracy change (pp) | Gained | Lost | Exact McNemar p-value |
| :--- | :--- | ---: | ---: | ---: | ---: |
| Turning dial down: 1 pass vs 2 passes (default) | strict | -99.0 pp | +0 | -198 | `4.98e-60` |
| Turning dial down: 1 pass vs 2 passes (default) | forced | -96.0 pp | +0 | -192 | `3.19e-58` |
| Turning dial up: 3 passes vs 2 passes (default) | strict | -6.0 pp | +1 | -13 | `0.0018` |
| Turning dial up: 3 passes vs 2 passes (default) | forced | -4.0 pp | +1 | -9 | `0.0215` |
| Turning dial up: 4 passes vs 2 passes (default) | strict | -11.0 pp | +0 | -22 | `4.77e-07` |
| Turning dial up: 4 passes vs 2 passes (default) | forced | -7.0 pp | +0 | -14 | `1.22e-04` |
| Default Loop (2 passes) vs plain Qwen3.5-9B | strict | +14.0 pp | +29 | -1 | `5.77e-08` |
| Default Loop (2 passes) vs plain Qwen3.5-9B | forced | +3.5 pp | +8 | -1 | `0.0391` |

### Plain-English `MATH-500` Charts & Real Side-by-Side Trace (`assets/`)
Every chart is programmatically audited to contain zero raw LaTeX tokens (`\`, `$`, `{`, `}`, `\boxed`, `</think>`) or internal config jargon.

In [ ]:
from IPython.display import Image, display

subprocess.run([sys.executable, "scripts/generate_charts.py"], check=True)
for chart_name in [
    "hero_trace.png",
    "loop_architecture.png",
    "accuracy_vs_passes.png",
    "speed_vs_accuracy.png",
    "budget_curve.png",
    "paired_changes.png",
]:
    chart_path = REPO_ROOT / "assets" / chart_name
    if chart_path.exists():
        print(f"=== assets/{chart_name} ===")
        display(Image(filename=str(chart_path), width=900))

## 7. Phase B2 — Harder Follow-Up: `AIME 2024 + 2025` Olympiad Sweep (`N = 60`, `32,768` Tokens)
On `MATH-500`, default `2` passes hits a **`99.5%` (`199/200`) ceiling**, leaving only 1 unsolved problem out of 200 to test whether extra middle-layer passes (`n = 3, 4`) can rescue harder multi-step derivations once `2` passes has real headroom. Here we evaluate all 60 AIME 2024 + 2025 olympiad problems (`HuggingFaceH4/aime_2024` + `MathArena/aime_2025`) on the **identical `vllm==0.33.0` + PR `#64832` stack** at **`32,768` max new tokens** (`max_num_seqs=16`, nested `2k / 4k / 8k / 16k / 32k` budgets, 512-token Pass-2 forced answer). Results live cleanly under `results/aime/` without touching `MATH-500` files.

In [ ]:
RUN_AIME_LIVE = False  # Set True on Colab GPU (>= 80 GB VRAM) to run or resume the 60-problem 32k AIME sweep
if RUN_AIME_LIVE:
    vllm_py = ensure_vllm_python()
    if not (REPO_ROOT / "data" / "aime_2024_2025.json").exists():
        subprocess.run([sys.executable, "scripts/prepare_aime.py"], check=True)
    subprocess.run([
        vllm_py, "scripts/run_sweep.py",
        "--backend", "vllm",
        "--problems-json", "data/aime_2024_2025.json",
        "--n-problems", "60",
        "--max-new-tokens", "32768",
        "--forced-max-tokens", "512",
        "--max-num-seqs", "16",
        "--out-dir", "results/aime",
        "--seed", "0",
        "--conditions", "loop_n2", "qwen35_9b", "loop_n4", "loop_n3", "loop_n1",
    ], check=True)

In [ ]:
if (REPO_ROOT / "results" / "aime" / "loop_n2.jsonl").exists():
    subprocess.run([sys.executable, "scripts/stats.py", "--results-dir", "results/aime"], check=True)
    subprocess.run([
        sys.executable, "scripts/generate_charts.py",
        "--results-dir", "results/aime",
        "--also-compare-math", "results/summary.csv",
    ], check=True)

    aime_summary_df = pd.read_csv("results/aime/summary.csv")
    aime_paired_df = pd.read_csv("results/aime/paired_stats.csv")
    print(aime_summary_df[display_cols].to_string(index=False))
    print("\nAIME 2024+2025 Paired Exact McNemar Comparisons (strict metric):")
    aime_strict = aime_paired_df[aime_paired_df["metric"] == "strict"]
    print(aime_strict[["comparison", "diff_cmp_minus_ref_pp", "gained_by_cmp", "lost_by_cmp", "mcnemar_p_value"]].to_string(index=False))

    for chart_name in ["cover_5x2.png", "aime_vs_math500.png", "aime_budget_curve.png"]:
        chart_path = REPO_ROOT / "assets" / chart_name
        if chart_path.exists():
            print(f"=== assets/{chart_name} ===")
            display(Image(filename=str(chart_path), width=900))
else:
    print("No AIME JSONL files found in results/aime/ yet. Set RUN_AIME_LIVE = True on a Colab GPU.")

### Logged `AIME 2024 + 2025` `N = 60` Results Snapshot (from `results/aime/summary.csv`)

| Dial setting | Layer steps / token | Solved cleanly (`≤32k` tokens) | With cap forced answer | Hit 32k cap | Median tokens | Strict answers / GPU-hr |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Loop 1 pass (32 layers) | 32 | **0.0%** (0/60, CI [0.0%, 6.02%]) | **0.0%** (0/60) | 80.0% | 32,768 | 0.0 |
| Loop 2 passes (56 layers, default) | 56 | **73.33%** (44/60, CI [60.99%, 82.87%]) | **76.67%** (46/60) | 20.0% | 7,844 | 72.0 |
| Loop 3 passes (80 layers) | 80 | **66.67%** (40/60, CI [54.06%, 77.27%]) | **78.33%** (47/60) | 30.0% | 10,786 | 40.8 |
| Loop 4 passes (104 layers) | 104 | **38.33%** (23/60, CI [27.09%, 50.98%]) | **53.33%** (32/60) | 50.0% | 32,062 | 13.2 |
| Qwen3.5-9B (32 layers, no loop) | 32 | **63.33%** (38/60, CI [50.68%, 74.38%]) | **76.67%** (46/60) | 35.0% | 27,268 | 69.1 |

#### Paired Exact McNemar Tests (`AIME 2024 + 2025`, `N = 60` Identical Olympiad Problems, Two-Sided)

| Comparison | Metric | Accuracy change (pp) | Gained | Lost | Exact McNemar p-value |
| :--- | :--- | ---: | ---: | ---: | ---: |
| Turning dial down: 1 pass vs 2 passes (default) | strict | -73.3 pp | +0 | -44 | `1.14e-13` |
| Turning dial down: 1 pass vs 2 passes (default) | forced | -76.7 pp | +0 | -46 | `2.84e-14` |
| Turning dial up: 3 passes vs 2 passes (default) | strict | -6.7 pp | +6 | -10 | `0.4545` |
| Turning dial up: 3 passes vs 2 passes (default) | forced | +1.7 pp | +6 | -5 | `1.0000` |
| Turning dial up: 4 passes vs 2 passes (default) | strict | -35.0 pp | +2 | -23 | `1.94e-05` |
| Turning dial up: 4 passes vs 2 passes (default) | forced | -23.3 pp | +3 | -17 | `0.0026` |
| Default Loop (2 passes) vs plain Qwen3.5-9B | strict | +10.0 pp | +12 | -6 | `0.2379` |
| Default Loop (2 passes) vs plain Qwen3.5-9B | forced | +0.0 pp | +6 | -6 | `1.0000` |
| 1-pass Loop vs plain Qwen3.5-9B | strict | -63.3 pp | +0 | -38 | `7.28e-12` |
| 1-pass Loop vs plain Qwen3.5-9B | forced | -76.7 pp | +0 | -46 | `2.84e-14` |
| 3-pass Loop vs plain Qwen3.5-9B | strict | +3.3 pp | +7 | -5 | `0.7744` |
| 3-pass Loop vs plain Qwen3.5-9B | forced | +1.7 pp | +6 | -5 | `1.0000` |
| 4-pass Loop vs plain Qwen3.5-9B | strict | -25.0 pp | +3 | -18 | `0.0015` |
| 4-pass Loop vs plain Qwen3.5-9B | forced | -23.3 pp | +3 | -17 | `0.0026` |

## 8. Phase D — Launch the Side-by-Side Gradio Explorer (`app.py`)
Replays both **`MATH-500` (`N = 200`)** and **`AIME 2024+2025` (`N = 60`)** offline (zero GPU needed), or pass `--enable-live-gpu` on a Colab GPU runtime to also test custom math questions live.

In [ ]:
LAUNCH_GRADIO = False  # Set True to open the interactive side-by-side app
if LAUNCH_GRADIO:
    from app import ReplayStore, build_app, HEAD_META
    demo = build_app(store=ReplayStore(), enable_live_gpu=torch.cuda.is_available())
    demo.launch(head=HEAD_META, share=False)